 # Artificial Intelligence Technology and Application

 ## Deep Learning Lab Guide - Student Version

 # 1 TextCNN Sentiment Analysis

 ## 1.1 Introduction

 This exercise implements the TextCNN sentiment analysis, a classic case in the deep learning field.

 Steps:

 - Download the dataset (rt-polarity.pos, rt-polarity.neg)

 - Generate data for training and validation

 - Define the TextCNN model structure, train and evaluate

 - Use the model for prediction

 ## 1.3 Detailed Design and Implementation

 ### 1.3.1 Data Preparation

 `rt-polarity.pos` stores 5000 positive reviews, `rt-polarity.neg` stores 5000 negative reviews.

 Directory: `./data/`

 ### 1.3.2 Procedure

 **Step 1: Import the Python library and module and configure running information.**

In [1]:
import math
import numpy as np
import os
import random
from pathlib import Path
from easydict import EasyDict as edict

try:
    import mindspore
    import mindspore.dataset as ds
    import mindspore.nn as nn
    from mindspore import Tensor, context
    from mindspore.train.model import Model
    from mindspore.nn.metrics import Accuracy
    from mindspore.train.serialization import load_checkpoint, load_param_into_net
    from mindspore.train.callback import ModelCheckpoint, CheckpointConfig, LossMonitor, TimeMonitor
    import mindspore.ops as ops
    
    cfg = edict({
        'name': 'movie review',
        'pre_trained': False,
        'num_classes': 2,
        'batch_size': 64,
        'epoch_size': 4,
        'weight_decay': 3e-5,
        'data_path': './data/',
        'device_target': 'CPU',
        'device_id': 0,
        'keep_checkpoint_max': 1,
        'checkpoint_path': './ckpt/train_textcnn-4_149.ckpt',
        'word_len': 51,
        'vec_length': 40
    })

    if not os.path.exists(cfg.data_path) or not any('rt-polarity' in f for f in os.listdir(cfg.data_path)):
        import urllib.request
        import zipfile
        import shutil
        print("Downloading TextCNN dataset from Huawei OBS...")
        os.makedirs(cfg.data_path, exist_ok=True)
        zip_path = os.path.join(cfg.data_path, "TextCNN.zip")
        urllib.request.urlretrieve("https://certification-data.obs.cn-north-4.myhuaweicloud.com/ENG/HCIA-AI/V3.5/chapter4/TextCNN.zip", zip_path)
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(cfg.data_path)
        
        # Flatten directory if extracted into a subfolder
        for root, dirs, files in os.walk(cfg.data_path):
            if 'rt-polarity.pos' in files and root != os.path.abspath(cfg.data_path):
                shutil.copy(os.path.join(root, 'rt-polarity.pos'), os.path.join(cfg.data_path, 'rt-polarity.pos'))
                shutil.copy(os.path.join(root, 'rt-polarity.neg'), os.path.join(cfg.data_path, 'rt-polarity.neg'))
        
        print("TextCNN dataset meticulously downloaded and safely extracted.")

    context.set_context(mode=context.GRAPH_MODE, device_target=cfg.device_target, device_id=cfg.device_id)
except ImportError:
    print("MindSpore is not installed.")


[WARNING] ME(78373:138917057564800,MainProcess):2026-04-03-05:33:38.761.000 [mindspore/context.py:1334] For 'context.set_context', the parameter 'device_target' will be deprecated and removed in a future version. Please use the api mindspore.set_device() instead.
[WARNING] ME(78373:138917057564800,MainProcess):2026-04-03-05:33:38.761.000 [mindspore/context.py:1334] For 'context.set_context', the parameter 'device_id' will be deprecated and removed in a future version. Please use the api mindspore.set_device() instead.


 **Step 2: Read and process data.**

In [2]:
try:
    class Generator():
        def __init__(self, input_list):
            self.input_list = input_list
        def __getitem__(self, item):
            return (np.array(self.input_list[item][0], dtype=np.int32),
                    np.array(self.input_list[item][1], dtype=np.int32))
        def __len__(self):
            return len(self.input_list)

    class MovieReview:
        def __init__(self, root_dir, maxlen, split):
            self.path = root_dir
            self.feelMap = {'neg': 0, 'pos': 1}
            self.files = []
            self.doConvert = False
            mypath = Path(self.path)
            
            if mypath.exists() and mypath.is_dir():
                self.files = [os.path.join(self.path, f) for f in os.listdir(self.path) if 'rt-polarity' in f]
            
            self.word_num = 0
            self.maxlen = 0
            self.minlen = float("inf")
            self.Pos = []
            self.Neg = []
            self.Vocab = {'<UNK>': 0}
            
            for filename in self.files:
                self.read_data(filename)

            self.text2vec(maxlen=maxlen)
            if self.Pos and self.Neg:
                self.split_dataset(split=split)

        def read_data(self, filePath):
            with open(filePath, 'r', encoding='utf-8', errors='ignore') as f:
                for sentence in f.readlines():
                    sentence = sentence.lower().strip()
                    chars_to_remove = ['\n', '"', "'", '.', ',', '[', ']', '(', ')', ':', '-', '\\', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '=', '$', '/', '*', ';', '<b>', '%']
                    for c in chars_to_remove:
                        sentence = sentence.replace(c, ' ')
                    
                    sentence = sentence.split(' ')
                    sentence = list(filter(lambda x: x, sentence))
                    
                    if sentence:
                        self.word_num += len(sentence)
                        self.maxlen = max(self.maxlen, len(sentence))
                        self.minlen = min(self.minlen, len(sentence))
                        
                        for word in sentence:
                            if word not in self.Vocab:
                                self.Vocab[word] = len(self.Vocab)
                                
                        if 'pos' in filePath:
                            self.Pos.append([sentence, self.feelMap['pos']])
                        else:
                            self.Neg.append([sentence, self.feelMap['neg']])

        def text2vec(self, maxlen):
            for SentenceLabel in self.Pos + self.Neg:
                vector = [0] * maxlen
                for index, word in enumerate(SentenceLabel[0]):
                    if index >= maxlen:
                        break
                    vector[index] = self.Vocab.get(word, 0)
                SentenceLabel[0] = vector
            self.doConvert = True

        def split_dataset(self, split):
            trunk_pos_size = math.ceil((1 - split) * len(self.Pos))
            trunk_neg_size = math.ceil((1 - split) * len(self.Neg))
            trunk_num = int(1 / (1 - split)) if split != 1 else 1
            pos_temp = []
            neg_temp = []
            
            for index in range(trunk_num):
                pos_temp.append(self.Pos[index * trunk_pos_size:(index + 1) * trunk_pos_size])
                neg_temp.append(self.Neg[index * trunk_neg_size:(index + 1) * trunk_neg_size])
                
            if len(pos_temp) > 2 and len(neg_temp) > 2:
                self.test = pos_temp.pop(2) + neg_temp.pop(2)
            else:
                self.test = pos_temp[-1] + neg_temp[-1] if pos_temp else []
                pos_temp = pos_temp[:-1]
                neg_temp = neg_temp[:-1]
                
            self.train = [i for item in pos_temp + neg_temp for i in item]
            random.shuffle(self.train)

        def get_dict_len(self):
            return len(self.Vocab) if self.doConvert else -1

        def create_train_dataset(self, epoch_size, batch_size):
            dataset = ds.GeneratorDataset(source=Generator(input_list=self.train), column_names=["data", "label"], shuffle=False)
            dataset = dataset.batch(batch_size=batch_size, drop_remainder=True)
            dataset = dataset.repeat(epoch_size)
            return dataset

        def create_test_dataset(self, batch_size):
            dataset = ds.GeneratorDataset(source=Generator(input_list=self.test), column_names=["data", "label"], shuffle=False)
            dataset = dataset.batch(batch_size=batch_size, drop_remainder=True)
            return dataset

    instance = MovieReview(root_dir=cfg.data_path, maxlen=cfg.word_len, split=0.9)
    if instance.files:
        dataset_train = instance.create_train_dataset(batch_size=cfg.batch_size, epoch_size=cfg.epoch_size)
        dataset_test = instance.create_test_dataset(batch_size=cfg.batch_size)
        batch_num = dataset_train.get_dataset_size()
        
        vocab_size = instance.get_dict_len()
        print("vocab_size:{0}".format(vocab_size))
except NameError:
    pass


vocab_size:18245


 **Step 4: Define a TextCNN model.**

In [3]:
try:
    def _weight_variable(shape, factor=0.01):
        init_value = np.random.randn(*shape).astype(np.float32) * factor
        return Tensor(init_value)

    def make_conv_layer(kernel_size):
        weight_shape = (96, 1, *kernel_size)
        weight = _weight_variable(weight_shape)
        return nn.Conv2d(in_channels=1, out_channels=96, kernel_size=kernel_size, padding=0,
                         pad_mode="valid", weight_init=weight, has_bias=True)

    class TextCNN(nn.Cell):
        def __init__(self, vocab_len, word_len, num_classes, vec_length):
            super(TextCNN, self).__init__()
            self.vec_length = vec_length
            self.word_len = word_len
            self.num_classes = num_classes

            self.unsqueeze = ops.ExpandDims()
            self.embedding = nn.Embedding(vocab_len, self.vec_length, embedding_table='normal')

            self.layer1 = self.make_layer(kernel_height=3)
            self.layer2 = self.make_layer(kernel_height=4)
            self.layer3 = self.make_layer(kernel_height=5)

            self.concat = ops.Concat(1)
            self.fc = nn.Dense(96 * 3, self.num_classes)
            self.drop = nn.Dropout(keep_prob=0.5)

        def make_layer(self, kernel_height):
            return nn.SequentialCell([
                make_conv_layer((kernel_height, self.vec_length)),
                nn.ReLU(),
                nn.MaxPool2d(kernel_size=(self.word_len - kernel_height + 1, 1))
            ])

        def construct(self, x):
            x = self.embedding(x)
            x = self.unsqueeze(x, 1)
            
            x1 = self.layer1(x)
            x2 = self.layer2(x)
            x3 = self.layer3(x)
            
            # Flatten pooling results
            x1 = ops.Squeeze(2)(ops.Squeeze(3)(x1))
            x2 = ops.Squeeze(2)(ops.Squeeze(3)(x2))
            x3 = ops.Squeeze(2)(ops.Squeeze(3)(x3))
            
            x = self.concat((x1, x2, x3))
            x = self.drop(x)
            x = self.fc(x)
            return x

    if instance.files:
        net = TextCNN(vocab_len=vocab_size, word_len=cfg.word_len, num_classes=cfg.num_classes, vec_length=cfg.vec_length)
        print("TextCNN model compiled.")
except NameError:
    pass


[WARNING] ME(78373:138917057564800,MainProcess):2026-04-03-05:33:38.949.000 [mindspore/nn/layer/basic.py:178] For Dropout, this parameter `keep_prob` will be deprecated, please use `p` instead.


TextCNN model compiled.


 **Step 5: Define training parameters.**

In [4]:
try:
    if instance.files:
        opt = nn.Adam(net.trainable_params(), learning_rate=1e-3, weight_decay=cfg.weight_decay)
        loss = nn.SoftmaxCrossEntropyWithLogits(sparse=True, reduction='mean')
        
        model = Model(net, loss_fn=loss, optimizer=opt, metrics={'acc': Accuracy()})
        
        config_ck = CheckpointConfig(save_checkpoint_steps=int(batch_num/2), keep_checkpoint_max=cfg.keep_checkpoint_max)
        time_cb = TimeMonitor(data_size=batch_num)
        ckpoint_cb = ModelCheckpoint(prefix="train_textcnn", directory="./ckpt", config=config_ck)
        loss_cb = LossMonitor(per_print_times=100)
except NameError:
    pass


[WARNING] ME(78373:138917057564800,MainProcess):2026-04-03-05:33:38.974.000 [mindspore/common/_decorator.py:69] 'FusedSparseAdam' is deprecated from version 2.8.0 and will be removed in a future version.


 **Step 6: Start training.**

In [5]:
try:
    if instance.files:
        print("Starting TextCNN training...")
        model.train(cfg.epoch_size, dataset_train, callbacks=[time_cb, ckpoint_cb, loss_cb])
        print("Train success")
        
        metric = model.eval(dataset_test)
        print("Validation Result:", metric)
except NameError:
    pass


[WARNING] ME(78373:138917057564800,MainProcess):2026-04-03-05:33:39.660.00 [mindspore/nn/layer/basic.py:204] For Dropout, this parameter `keep_prob` will be deprecated, please use `p` instead.


Starting TextCNN training...


[ERROR] CORE(78373,7e5825de9080,python):2026-04-03-05:33:39.492.625 [mindspore/core/utils/file_utils.cc:253] GetRealPath] Get realpath failed, path[/tmp/ipykernel_78373/3353811085.py]
[WARNING] CORE(78373,7e5825de9080,python):2026-04-03-05:33:39.492.637 [mindspore/core/utils/info.cc:95] ReadSectionDebugInfoFromFile] The file '/tmp/ipykernel_78373/3353811085.py' may not exists.
[ERROR] CORE(78373,7e5825de9080,python):2026-04-03-05:33:39.537.467 [mindspore/core/utils/file_utils.cc:253] GetRealPath] Get realpath failed, path[/tmp/ipykernel_78373/3353811085.py]
[WARNING] CORE(78373,7e5825de9080,python):2026-04-03-05:33:39.537.481 [mindspore/core/utils/info.cc:95] ReadSectionDebugInfoFromFile] The file '/tmp/ipykernel_78373/3353811085.py' may not exists.
[ERROR] CORE(78373,7e5825de9080,python):2026-04-03-05:33:39.537.490 [mindspore/core/utils/file_utils.cc:253] GetRealPath] Get realpath failed, path[/tmp/ipykernel_78373/3353811085.py]
[WARNING] CORE(78373,7e5825de9080,python):2026-04-03-05:

epoch: 1 step: 100, loss is 0.5872827768325806
epoch: 1 step: 200, loss is 0.4005497992038727
epoch: 1 step: 300, loss is 0.2713679075241089
epoch: 1 step: 400, loss is 0.21835696697235107
epoch: 1 step: 500, loss is 0.13612376153469086
Train epoch time: 14819.151 ms, per step time: 24.864 ms
epoch: 2 step: 4, loss is 0.06732992827892303
epoch: 2 step: 104, loss is 0.05239786207675934
epoch: 2 step: 204, loss is 0.02015221305191517
epoch: 2 step: 304, loss is 0.03247940540313721
epoch: 2 step: 404, loss is 0.0899161770939827
epoch: 2 step: 504, loss is 0.011120187118649483
Train epoch time: 12937.544 ms, per step time: 21.707 ms
epoch: 3 step: 8, loss is 0.02538909949362278
epoch: 3 step: 108, loss is 0.008497340604662895
epoch: 3 step: 208, loss is 0.0018838654505088925
epoch: 3 step: 308, loss is 0.0036458708345890045
epoch: 3 step: 408, loss is 0.002678476506844163
epoch: 3 step: 508, loss is 0.002536775078624487
Train epoch time: 14835.585 ms, per step time: 24.892 ms
epoch: 4 step

[WARNING] ME(78373:138917057564800,MainProcess):2026-04-03-05:34:35.304.000 [mindspore/nn/layer/basic.py:204] For Dropout, this parameter `keep_prob` will be deprecated, please use `p` instead.


Train success


[ERROR] CORE(78373,7e5825de9080,python):2026-04-03-05:34:35.418.834 [mindspore/core/utils/file_utils.cc:253] GetRealPath] Get realpath failed, path[/tmp/ipykernel_78373/3353811085.py]
[WARNING] CORE(78373,7e5825de9080,python):2026-04-03-05:34:35.418.846 [mindspore/core/utils/info.cc:95] ReadSectionDebugInfoFromFile] The file '/tmp/ipykernel_78373/3353811085.py' may not exists.


Validation Result: {'acc': 0.75390625}


 **Step 7: Test and validate data.**

In [6]:
try:
    def preprocess(sentence):
        sentence = sentence.lower().strip()
        chars_to_remove = ['\n', '"', "'", '.', ',', '[', ']', '(', ')', ':', '-', '\\', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '=', '$', '/', '*', ';', '<b>', '%']
        for c in chars_to_remove:
            sentence = sentence.replace(c, ' ')
        sentence = sentence.split(' ')
        vector = [0] * cfg.word_len
        if instance.files:
            for index, word in enumerate(sentence):
                if index >= cfg.word_len:
                    break
                vector[index] = instance.Vocab.get(word, 0)
        return vector

    def inference(review_en):
        review_vec = preprocess(review_en)
        input_en = Tensor(np.array([review_vec]).astype(np.int32))
        output = net(input_en)
        if np.argmax(np.array(output[0])) == 1:
            print("Positive comments")
        else:
            print("Negative comments")

    if instance.files:
        review_en = "the movie is so boring"
        inference(review_en)
except NameError:
    pass


[WARNING] ME(78373:138917057564800,MainProcess):2026-04-03-05:34:35.730.000 [mindspore/nn/layer/basic.py:204] For Dropout, this parameter `keep_prob` will be deprecated, please use `p` instead.


Negative comments


[ERROR] CORE(78373,7e5825de9080,python):2026-04-03-05:34:35.841.577 [mindspore/core/utils/file_utils.cc:253] GetRealPath] Get realpath failed, path[/tmp/ipykernel_78373/3353811085.py]
[WARNING] CORE(78373,7e5825de9080,python):2026-04-03-05:34:35.841.592 [mindspore/core/utils/info.cc:95] ReadSectionDebugInfoFromFile] The file '/tmp/ipykernel_78373/3353811085.py' may not exists.
[ERROR] CORE(78373,7e5825de9080,python):2026-04-03-05:34:35.841.652 [mindspore/core/utils/file_utils.cc:253] GetRealPath] Get realpath failed, path[/tmp/ipykernel_78373/3353811085.py]
[WARNING] CORE(78373,7e5825de9080,python):2026-04-03-05:34:35.841.656 [mindspore/core/utils/info.cc:95] ReadSectionDebugInfoFromFile] The file '/tmp/ipykernel_78373/3353811085.py' may not exists.
